# Lab 03 - Semantic Search with Embeddings

**Knowledge base:** `knowledge_base/03_retrieval/04_semantic_search.md`

**Concepts:** Embedding at index time · Query embedding · Cosine ranking · Synonym handling · Pre-computed embeddings

This lab implements a full semantic search system. You embed documents once (index time),
then embed each query at search time and rank by cosine similarity.

---

## Setup

In [1]:
import numpy as np
import joblib
import os
from sentence_transformers import SentenceTransformer

MODEL_NAME = "BAAI/bge-base-en-v1.5"
print(f"Loading {MODEL_NAME}...")
model = SentenceTransformer(MODEL_NAME)
print("✅ Model ready")

def cosine_similarity(v1, matrix):
    v1 = np.array(v1, dtype=np.float32).ravel()
    M  = np.array(matrix, dtype=np.float32)
    if M.ndim == 1:
        M = M.reshape(1, -1)
    v1_norm = np.linalg.norm(v1)
    M_norms = np.linalg.norm(M, axis=1)
    denom   = v1_norm * M_norms
    return (M @ v1 / np.where(denom == 0, 1.0, denom)).tolist()

Loading BAAI/bge-base-en-v1.5...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

✅ Model ready


---
## 1 - Index time: embed all documents once

In production, you embed every document when it enters the database.
At query time you only embed the query - never re-embed all documents.

In [2]:
# A small knowledge base covering RAG, search, and Egypt
knowledge_base = [
    "RAG stands for Retrieval Augmented Generation - it combines search with LLM generation.",
    "BM25 is a keyword-based ranking algorithm used in information retrieval.",
    "Cosine similarity measures the angle between two vectors in high-dimensional space.",
    "The Suez Canal was inaugurated in 1869 and is 193 kilometres long.",
    "Cairo is the capital of Egypt, located along the Nile River.",
    "Embeddings are dense numerical representations of text learned by transformer models.",
    "Hybrid search combines keyword search and semantic search for better retrieval.",
    "Metadata filtering applies boolean conditions to exclude documents by attribute.",
    "The Great Pyramid of Giza is one of the Seven Wonders of the Ancient World.",
    "Sentence transformers are models fine-tuned to produce semantically meaningful embeddings.",
    "Reciprocal Rank Fusion (RRF) merges multiple ranked lists into a single ranking.",
    "Vector databases store embeddings and support fast approximate nearest-neighbor search.",
    "Precision@K measures what fraction of the top-K retrieved documents are relevant.",
    "Recall@K measures what fraction of all relevant documents appear in the top-K results.",
    "Egypt has a population of approximately 105 million and its official language is Arabic.",
]

print("Embedding knowledge base at index time...")
doc_embeddings = model.encode(knowledge_base, show_progress_bar=True)
print(f"\n✅ Indexed {len(knowledge_base)} documents")
print(f"   Embedding matrix shape: {doc_embeddings.shape}  ← (documents, dimensions)")

Embedding knowledge base at index time...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


✅ Indexed 15 documents
   Embedding matrix shape: (15, 768)  ← (documents, dimensions)


In [3]:
# Save to disk - in production you would store this in a vector database
joblib.dump(doc_embeddings, "doc_embeddings.joblib")
print("✅ Embeddings saved to doc_embeddings.joblib")
print("   In production: upload to Weaviate, Pinecone, or FAISS instead")

✅ Embeddings saved to doc_embeddings.joblib
   In production: upload to Weaviate, Pinecone, or FAISS instead


---
## 2 - Query time: embed query and rank by similarity

In [4]:
def semantic_search(query: str, documents: list, doc_embeddings: np.ndarray,
                     top_k: int = 3) -> list:
    '''
    Semantic search over pre-embedded documents.

    Args:
        query:          The search query.
        documents:      The original document strings.
        doc_embeddings: Pre-computed embedding matrix (n_docs, n_dims).
        top_k:          Number of results to return.

    Returns:
        List of (score, document) tuples sorted by cosine similarity descending.
    '''
    # Embed the query - this is the ONLY embedding call at search time
    query_emb = model.encode(query)

    # Rank all documents by cosine similarity to the query
    scores = cosine_similarity(query_emb, doc_embeddings)

    # Sort and return top-k
    ranked = sorted(zip(scores, documents), reverse=True)
    return ranked[:top_k]


# Test with a direct query
query = "How does retrieval work in RAG?"
print(f"Query: {query}\n")
for score, doc in semantic_search(query, knowledge_base, doc_embeddings, top_k=3):
    print(f"  [{score:.4f}] {doc}")

Query: How does retrieval work in RAG?

  [0.8147] RAG stands for Retrieval Augmented Generation - it combines search with LLM generation.
  [0.6241] Hybrid search combines keyword search and semantic search for better retrieval.
  [0.5799] Recall@K measures what fraction of all relevant documents appear in the top-K results.


---
## 3 - Semantic search handles synonyms and paraphrases

This is the core advantage over BM25. No shared words required.

In [5]:
test_queries = [
    # Direct query - should also work with BM25
    "What is the Suez Canal?",
    # Paraphrase - different words, same meaning
    "Tell me about the waterway that connects the Red Sea and the Mediterranean",
    # Synonym - 'vector representations' instead of 'embeddings'
    "What are vector representations of text?",
    # Conceptual - no keywords from any document
    "How do you measure whether two sentences mean the same thing?",
]

for q in test_queries:
    print(f"Query: {q}")
    results = semantic_search(q, knowledge_base, doc_embeddings, top_k=2)
    for score, doc in results:
        print(f"  [{score:.4f}] {doc}")
    print()

Query: What is the Suez Canal?
  [0.7724] The Suez Canal was inaugurated in 1869 and is 193 kilometres long.
  [0.6508] Cairo is the capital of Egypt, located along the Nile River.

Query: Tell me about the waterway that connects the Red Sea and the Mediterranean
  [0.6560] The Suez Canal was inaugurated in 1869 and is 193 kilometres long.
  [0.6211] Cairo is the capital of Egypt, located along the Nile River.

Query: What are vector representations of text?
  [0.7446] Embeddings are dense numerical representations of text learned by transformer models.
  [0.7013] Vector databases store embeddings and support fast approximate nearest-neighbor search.

Query: How do you measure whether two sentences mean the same thing?
  [0.5694] Sentence transformers are models fine-tuned to produce semantically meaningful embeddings.
  [0.5469] Cosine similarity measures the angle between two vectors in high-dimensional space.



---
## 4 - Comparing keyword vs semantic search

Let's see where each method wins and where it fails.

In [6]:
from rank_bm25 import BM25Okapi
import subprocess
subprocess.run(["pip", "install", "rank-bm25", "--break-system-packages", "-q"], capture_output=True)

tokenized_kb = [doc.lower().split() for doc in knowledge_base]
bm25         = BM25Okapi(tokenized_kb)

def keyword_search(query: str, documents: list, top_k: int = 3) -> list:
    scores = bm25.get_scores(query.lower().split())
    ranked = sorted(zip(scores, documents), reverse=True)
    return ranked[:top_k]


# Compare on synonym query - keyword should fail, semantic should succeed
synonym_query = "vector representations of text"

print(f"Query: '{synonym_query}'\n")
print("BM25 keyword search:")
for score, doc in keyword_search(synonym_query, knowledge_base, top_k=3):
    print(f"  [{score:.4f}] {doc}")

print()
print("Semantic search:")
for score, doc in semantic_search(synonym_query, knowledge_base, doc_embeddings, top_k=3):
    print(f"  [{score:.4f}] {doc}")

Query: 'vector representations of text'

BM25 keyword search:
  [5.0340] Embeddings are dense numerical representations of text learned by transformer models.
  [2.4188] Vector databases store embeddings and support fast approximate nearest-neighbor search.
  [0.5893] The Great Pyramid of Giza is one of the Seven Wonders of the Ancient World.

Semantic search:
  [0.7784] Embeddings are dense numerical representations of text learned by transformer models.
  [0.7582] Vector databases store embeddings and support fast approximate nearest-neighbor search.
  [0.6885] Cosine similarity measures the angle between two vectors in high-dimensional space.


In [7]:
# Compare on technical term query - keyword should win, semantic may struggle
technical_query = "BM25 algorithm"

print(f"Query: '{technical_query}'\n")
print("BM25 keyword search:")
for score, doc in keyword_search(technical_query, knowledge_base, top_k=3):
    print(f"  [{score:.4f}] {doc}")

print()
print("Semantic search:")
for score, doc in semantic_search(technical_query, knowledge_base, doc_embeddings, top_k=3):
    print(f"  [{score:.4f}] {doc}")

print()
print("Insight: keyword search excels at exact technical terms.")
print("Semantic search excels at paraphrases and conceptual queries.")
print("This is exactly why hybrid search (lab04) combines both.")

Query: 'BM25 algorithm'

BM25 keyword search:
  [4.8376] BM25 is a keyword-based ranking algorithm used in information retrieval.
  [0.0000] Vector databases store embeddings and support fast approximate nearest-neighbor search.
  [0.0000] The Suez Canal was inaugurated in 1869 and is 193 kilometres long.

Semantic search:
  [0.8147] BM25 is a keyword-based ranking algorithm used in information retrieval.
  [0.6331] Vector databases store embeddings and support fast approximate nearest-neighbor search.
  [0.6110] Cosine similarity measures the angle between two vectors in high-dimensional space.

Insight: keyword search excels at exact technical terms.
Semantic search excels at paraphrases and conceptual queries.
This is exactly why hybrid search (lab04) combines both.


---
## 5 - Exercise: build a semantic RAG pipeline

In [ ]:
import sys
sys.path.insert(0, ".")
from dotenv import load_dotenv
from utils import generate_with_single_input
load_dotenv(override=True)

def semantic_rag(query: str, documents: list, doc_embeddings: np.ndarray,
                 top_k: int = 3) -> str:
    '''
    Full semantic RAG: embed query -> cosine rank -> augment prompt -> generate.

    YOUR TASK: implement this function using semantic_search() and generate_with_single_input().
    '''
    # Step 1: retrieve
    # YOUR CODE HERE

    # Step 2: build augmented prompt
    # YOUR CODE HERE

    # Step 3: generate
    # YOUR CODE HERE
    pass


# Test your function
test_q = "What is Reciprocal Rank Fusion?"
answer = semantic_rag(test_q, knowledge_base, doc_embeddings)
print(f"Q: {test_q}")
print(f"A: {answer}")

assert answer is not None and len(answer) > 0
print("\n✅ Exercise passed")

---
**Lab 03 complete.**

You have a working semantic search system: embed once at index time, query at search time.

Strengths: synonyms, paraphrases, conceptual queries.
Weakness: exact technical terms, product names, IDs.

**Next:** `lab04_hybrid_search.ipynb` - combine keyword and semantic search with RRF.